# White-box adversarial training

This notebook trains:
1. a clean ResNet-18 layer3 wrapper model,
2. an input-level PGD adversarially trained model,
3. a latent-level PGD adversarially trained model,
4. a combined input + latent PGD adversarially trained model.

In [ ]:
# Imports
from xml.parsers.expat import model

import os
import csv
import pickle

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

import torch
from torch import nn
from torch.utils.data import Dataset, TensorDataset, DataLoader
from torchvision.models import resnet18, ResNet18_Weights
import torchvision.transforms.functional as F

import utils

In [ ]:
# Configuration
# In a .py script, BASE_DIR was os.path.dirname(os.path.abspath(__file__)).
# In a notebook, use the current working directory or set this manually.
BASE_DIR = os.getcwd()

CACHE_PATH = os.path.join(BASE_DIR, "gtsrb_cache.pkl")

num_classes = 43
batch_size_train = 64
num_workers = 4
pin_memory = True

clean_epochs = 10
adv_epochs = 15

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Load cached training data
print("Reading training data...")

with open(CACHE_PATH, "rb") as f:
    trainImages, trainLabels = pickle.load(f)

train_dataset = utils.GTSRBDataset(trainImages, trainLabels, target_size=224)
print(f"Total training samples: {len(train_dataset)}")

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size_train,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=pin_memory,
)

In [ ]:
# Shared training objects
criterion = nn.CrossEntropyLoss()

def make_model():
    model = utils.ResNet18_L3(num_classes=num_classes)
    return model.to(device)

def make_optimizer(model):
    return torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)

## 1. Clean training

In [ ]:
# Clean training
model = make_model()
optimizer = make_optimizer(model)

for epoch in range(clean_epochs):
    training_correct = 0
    model.train()

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(utils.normalize_batch(images))
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        _, predicted = torch.max(outputs.data, 1)
        training_correct += (predicted == labels).sum().item()

    print(
        f"Clean Epoch {epoch + 1}/{clean_epochs}, "
        f"Loss: {loss.item():.4f}, "
        f"Accuracy: {100 * training_correct / len(train_dataset):.2f}%"
    )

clean_checkpoint_path = os.path.join(BASE_DIR, "resnet18_gtsrb_clean.pth")
torch.save(model.state_dict(), clean_checkpoint_path)
print(f"Clean training complete. Model saved as {clean_checkpoint_path}")

## 2. Input-level PGD adversarial training

In [ ]:
# Input-level PGD adversarial training
model = make_model()
model.load_state_dict(torch.load(clean_checkpoint_path, map_location=device))
optimizer = make_optimizer(model)

lambda_clean = 1 / 2
lambda_input_adv = 1 / 2

for epoch in range(adv_epochs):
    training_correct = 0
    model.train()

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        clean_outputs = model(utils.normalize_batch(images))
        clean_loss = criterion(clean_outputs, labels)

        adv_input_images = utils.generate_adversarial_batch(
            model=model,
            images=images,
            labels=labels,
            attack="PGD",
            device=device,
            epsilon=8 / 255.0,
            alpha=2 / 255.0,
            num_steps=10,
            criterion=criterion,
        )

        adv_input_outputs = model(utils.normalize_batch(adv_input_images))
        adv_input_loss = criterion(adv_input_outputs, labels)

        loss = (lambda_clean * clean_loss) + (lambda_input_adv * adv_input_loss)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        _, predicted = torch.max(adv_input_outputs.data, 1)
        training_correct += (predicted == labels).sum().item()

    print(
        f"Input-PGD Epoch {epoch + 1}/{adv_epochs}, "
        f"Loss: {loss.item():.4f}, "
        f"Accuracy: {100 * training_correct / len(train_dataset):.2f}%"
    )

input_adv_checkpoint_path = os.path.join(BASE_DIR, "resnet18_gtsrb_input_adv.pth")
torch.save(model.state_dict(), input_adv_checkpoint_path)
print(f"Saved input-PGD robust model to {input_adv_checkpoint_path}")

## 3. Latent-level PGD adversarial training

In [ ]:
# Latent-level PGD adversarial training
model = make_model()
model.load_state_dict(torch.load(clean_checkpoint_path, map_location=device))
optimizer = make_optimizer(model)

epsilon_latent = 0.2
lambda_latent_adv = 1 / 2
lambda_clean = 1 / 2

for epoch in range(adv_epochs):
    training_correct = 0
    model.train()

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        clean_outputs = model(utils.normalize_batch(images))
        clean_loss = criterion(clean_outputs, labels)

        z_adv = utils.pgd_attack_latent_l3(
            model=model,
            image=images,
            label=labels,
            epsilon=epsilon_latent,
            alpha=epsilon_latent / 4.0,
            num_steps=5,
            criterion=criterion,
            device=device,
        )

        adv_latent_outputs = model.decode(z_adv)
        adv_latent_loss = criterion(adv_latent_outputs, labels)

        loss = (lambda_clean * clean_loss) + (lambda_latent_adv * adv_latent_loss)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        _, predicted = torch.max(adv_latent_outputs.data, 1)
        training_correct += (predicted == labels).sum().item()

    print(
        f"Latent-PGD Epoch {epoch + 1}/{adv_epochs}, "
        f"Loss: {loss.item():.4f}, "
        f"Accuracy: {100 * training_correct / len(train_dataset):.2f}%"
    )

latent_adv_checkpoint_path = os.path.join(BASE_DIR, "resnet18_gtsrb_latent_adv.pth")
torch.save(model.state_dict(), latent_adv_checkpoint_path)
print(f"Saved latent-PGD robust model to {latent_adv_checkpoint_path}")

## 4. Combined input-level PGD + latent-level PGD adversarial training

In [ ]:
# Combined input-level PGD and latent-level PGD adversarial training
model = make_model()
model.load_state_dict(torch.load(clean_checkpoint_path, map_location=device))
optimizer = make_optimizer(model)

epsilon_latent = 0.2
lambda_latent_adv = 1 / 3
lambda_clean = 1 / 3
lambda_input_adv = 1 / 3

for epoch in range(adv_epochs):
    training_correct = 0
    model.train()

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        clean_outputs = model(utils.normalize_batch(images))
        clean_loss = criterion(clean_outputs, labels)

        adv_input_images = utils.generate_adversarial_batch(
            model=model,
            images=images,
            labels=labels,
            attack="PGD",
            device=device,
            epsilon=8 / 255.0,
            alpha=2 / 255.0,
            num_steps=10,
            criterion=criterion,
        )

        adv_input_outputs = model(utils.normalize_batch(adv_input_images))
        adv_input_loss = criterion(adv_input_outputs, labels)

        z_adv = utils.pgd_attack_latent_l3(
            model=model,
            image=images,
            label=labels,
            epsilon=epsilon_latent,
            alpha=epsilon_latent / 4.0,
            num_steps=5,
            criterion=criterion,
            device=device,
        )

        adv_latent_outputs = model.decode(z_adv)
        adv_latent_loss = criterion(adv_latent_outputs, labels)

        loss = (
            (lambda_clean * clean_loss)
            + (lambda_input_adv * adv_input_loss)
            + (lambda_latent_adv * adv_latent_loss)
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        _, predicted = torch.max(adv_input_outputs.data, 1)
        training_correct += (predicted == labels).sum().item()

    print(
        f"Input+Latent-PGD Epoch {epoch + 1}/{adv_epochs}, "
        f"Loss: {loss.item():.4f}, "
        f"Accuracy: {100 * training_correct / len(train_dataset):.2f}%"
    )

input_latent_adv_checkpoint_path = os.path.join(BASE_DIR, "resnet18_gtsrb_input_latent_adv.pth")
torch.save(model.state_dict(), input_latent_adv_checkpoint_path)
print(f"Saved input-latent-PGD robust model to {input_latent_adv_checkpoint_path}")